# Software Development Trends in Algeria

This notebook tracks GitHub software-development activity in Algeria over time and compares Algeria with the same peer groups used in the AI adoption notebook:

- Structural peers: Ecuador, Peru, Ghana, Vietnam, Colombia.
- Regional peers: Egypt, Jordan, Morocco, Tunisia, Iraq.
- Aspirational peers: Malaysia, Chile, Poland, Romania.

The analysis uses quarterly GitHub Innovation Graph data stored locally under `data/github` and annual population statistics from the World Bank `SP.POP.TOTL` indicator.

## Setup

In [81]:
from pathlib import Path
from urllib.request import urlopen
import json

import altair as alt
import pandas as pd
import pycountry
import attaviz

attaviz.enable()
alt.data_transformers.enable("default", max_rows=None)

DataTransformerRegistry.enable('default')

In [82]:
def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
GITHUB_DIR = DATA_DIR / "github"

COUNTRY_CODE = "DZ"
COUNTRY_NAME = "Algeria"

PEER_GROUPS = {
    "Structural": ["EC", "PE", "GH", "VN", "CO"],
    "Regional": ["EG", "JO", "MA", "TN", "IQ"],
    "Aspirational": ["MY", "CL", "PL", "RO"],
}

COUNTRY_NAMES = {
    "DZ": "Algeria",
    "EG": "Egypt",
    "JO": "Jordan",
    "MA": "Morocco",
    "TN": "Tunisia",
    "IQ": "Iraq",
    "EC": "Ecuador",
    "PE": "Peru",
    "GH": "Ghana",
    "VN": "Vietnam",
    "CO": "Colombia",
    "MY": "Malaysia",
    "CL": "Chile",
    "PL": "Poland",
    "RO": "Romania",
}

PEER_CODES = [COUNTRY_CODE] + [code for codes in PEER_GROUPS.values() for code in codes]
ALGERIA_COLOR = "#0071BC"
PEER_COLORS = ["#8A969F", "#9FA8AF", "#B0B8BE", "#C1C8CD", "#D2D7DB"]

ISO2_TO_ISO3 = {
    "DZ": "DZA",
    "EG": "EGY",
    "JO": "JOR",
    "MA": "MAR",
    "TN": "TUN",
    "IQ": "IRQ",
    "EC": "ECU",
    "PE": "PER",
    "GH": "GHA",
    "VN": "VNM",
    "CO": "COL",
    "MY": "MYS",
    "CL": "CHL",
    "PL": "POL",
    "RO": "ROU",
}
ISO3_TO_ISO2 = {iso3: iso2 for iso2, iso3 in ISO2_TO_ISO3.items()}

def country_name_from_iso2(code: str) -> str:
    if code == "EU":
        return "European Union"
    if code in COUNTRY_NAMES:
        return COUNTRY_NAMES[code]
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else code


## Data

In [83]:
import ssl
def load_world_bank_population(iso2_codes: list[str]) -> pd.DataFrame:
    url = (
        "https://api.worldbank.org/v2/country/"
        + ";".join(ISO2_TO_ISO3[code] for code in iso2_codes)
        + "/indicator/SP.POP.TOTL?format=json&per_page=20000"
    )
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    with urlopen(url, context=ctx) as response:
        payload = json.load(response)

    return (
        pd.DataFrame(payload[1])
        .loc[:, ["countryiso3code", "date", "value"]]
        .rename(
            columns={
                "countryiso3code": "iso3_code",
                "date": "year",
                "value": "population",
            }
        )
        .dropna(subset=["population"])
        .assign(
            iso2_code=lambda d: d["iso3_code"].map(ISO3_TO_ISO2),
            year=lambda d: d["year"].astype(int),
            population=lambda d: d["population"].astype(float),
        )
        .dropna(subset=["iso2_code"])
        .sort_values(["iso2_code", "year"])
    )


def attach_latest_population(github_df: pd.DataFrame, population_df: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for iso2_code, country_df in github_df.sort_values("year").groupby("iso2_code", sort=False):
        pop_country = population_df.loc[population_df["iso2_code"] == iso2_code].sort_values("year")
        pieces.append(
            pd.merge_asof(
                country_df.sort_values("year"),
                pop_country[["year", "population"]],
                on="year",
                direction="backward",
            )
        )
    return pd.concat(pieces, ignore_index=True)


def load_github_metric(metric: str, population_df: pd.DataFrame) -> pd.DataFrame:
    raw = (
        pd.read_csv(GITHUB_DIR / f"{metric}.csv")
        .loc[lambda d: d["iso2_code"].isin(PEER_CODES)]
        .assign(
            quarter_start=lambda d: pd.PeriodIndex(
                d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
            ).to_timestamp(),
            country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES),
        )
        .sort_values(["iso2_code", "year", "quarter"])
    )

    per_100k_col = f"{metric}_per_100k"
    return (
        attach_latest_population(raw, population_df)
        .assign(**{per_100k_col: lambda d: d[metric] / d["population"] * 100_000})
        .dropna(subset=[per_100k_col])
        .sort_values(["iso2_code", "year", "quarter"])
    )


population = load_world_bank_population(PEER_CODES)

developers = load_github_metric("developers", population)
repositories = load_github_metric("repositories", population)
organizations = load_github_metric("organizations", population)
git_pushes = load_github_metric("git_pushes", population)

algeria_trend = developers.loc[developers["iso2_code"] == COUNTRY_CODE].copy()
algeria_summary = {
    "first_quarter": f"{int(algeria_trend.iloc[0]['year'])} Q{int(algeria_trend.iloc[0]['quarter'])}",
    "last_quarter": f"{int(algeria_trend.iloc[-1]['year'])} Q{int(algeria_trend.iloc[-1]['quarter'])}",
    "first_developers_per_100k": algeria_trend.iloc[0]["developers_per_100k"],
    "last_developers_per_100k": algeria_trend.iloc[-1]["developers_per_100k"],
    "growth_multiple": algeria_trend.iloc[-1]["developers_per_100k"] / algeria_trend.iloc[0]["developers_per_100k"],
}

algeria_summary

{'first_quarter': '2020 Q1',
 'last_quarter': '2025 Q4',
 'first_developers_per_100k': np.float64(208.48011053789432),
 'last_developers_per_100k': np.float64(1138.2908832060489),
 'growth_multiple': np.float64(5.459949537963948)}

## GitHub Developers Per 100k People Over Time

Algeria's visible GitHub developer base grew substantially in per-capita terms over the Innovation Graph period. The panels below compare Algeria with each peer group separately, using annual World Bank population statistics to express quarterly GitHub developer counts as developers per 100k people.

In [84]:
def make_peer_trend_panel(
    data: pd.DataFrame,
    raw_col: str,
    value_col: str,
    group_name: str,
    y_axis_title: str,
    width: int = 150,
    height: int = 100,
    show_y_title: bool = False,
) -> alt.Chart:
    codes = [COUNTRY_CODE] + PEER_GROUPS[group_name]
    d = data.loc[data["iso2_code"].isin(codes)].copy()
    latest_quarter = d["quarter_start"].max()
    labels = d.loc[d["quarter_start"] == latest_quarter].copy()

    color_domain = [COUNTRY_NAMES[code] for code in codes]
    color_range = [ALGERIA_COLOR] + PEER_COLORS[: len(codes) - 1]

    y_title = y_axis_title if show_y_title else None

    base = alt.Chart(d).encode(
        x=alt.X("quarter_start:T", title=None),
        y=alt.Y(
            f"{value_col}:Q",
            title=y_title,
            axis=alt.Axis(format="~s"),
            scale=alt.Scale(zero=True),
        ),
        color=alt.Color(
            "country_name:N",
            title=None,
            scale=alt.Scale(domain=color_domain, range=color_range),
            legend=None,
        ),
        detail="country_name:N",
    )

    lines = base.mark_line().encode(
        strokeWidth=alt.condition(
            alt.datum.iso2_code == COUNTRY_CODE, alt.value(4), alt.value(2)
        ),
        opacity=alt.condition(
            alt.datum.iso2_code == COUNTRY_CODE, alt.value(1.0), alt.value(0.72)
        ),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("quarter:O", title="Quarter"),
            alt.Tooltip(f"{value_col}:Q", title=y_axis_title, format=",.0f"),
            alt.Tooltip(f"{raw_col}:Q", title=raw_col.replace("_", " ").title(), format=","),
            alt.Tooltip("population:Q", title="Population", format=",.0f"),
        ],
    )

    points = base.transform_filter(alt.datum.iso2_code == COUNTRY_CODE).mark_point(
        size=38, filled=True, stroke="white", strokeWidth=1
    )

    latest_labels = (
        alt.Chart(labels)
        .mark_text(align="left", baseline="middle", dx=6, fontSize=11)
        .encode(
            x=alt.X("quarter_start:T"),
            y=alt.Y(f"{value_col}:Q"),
            text="country_name:N",
            color=alt.Color(
                "country_name:N",
                scale=alt.Scale(domain=color_domain, range=color_range),
                legend=None,
            ),
        )
    )

    return (lines + points + latest_labels).properties(
        width=width,
        height=height,
        title=alt.Title(
            text=f"{group_name} peers",
            anchor="start",
            fontSize=12,
            subtitleFontSize=10,
            offset=8,
        ),
    )


def make_three_peer_trends(
    data: pd.DataFrame,
    raw_col: str,
    value_col: str,
    title: str,
    subtitle: str,
    y_axis_title: str,
) -> alt.Chart:
    return alt.hconcat(
        make_peer_trend_panel(data, raw_col, value_col, "Structural", y_axis_title, show_y_title=True),
        make_peer_trend_panel(data, raw_col, value_col, "Regional", y_axis_title),
        make_peer_trend_panel(data, raw_col, value_col, "Aspirational", y_axis_title),
        spacing=28,
    ).resolve_scale(y="independent").properties(
        title=alt.Title(
            text=title,
            subtitle=subtitle,
            fontSize=14,
            subtitleFontSize=11,
            offset=10,
        )
    )


trend_panels = make_three_peer_trends(
    developers,
    raw_col="developers",
    value_col="developers_per_100k",
    title="GitHub Developers per 100k People in Algeria and Peer Countries",
    subtitle="Quarterly GitHub developers per 100k people, 2020 Q1-2025 Q4",
    y_axis_title="Developers per 100k people",
)

attaviz.add_caption(trend_panels, "Source: GitHub Innovation Graph data; World Bank SP.POP.TOTL")

alt.VConcatChart(...)

The comparison is normalized by population, using annual World Bank total population (`SP.POP.TOTL`). For GitHub quarters that are more recent than the latest available World Bank population year, the latest available annual population is carried forward. Algeria is shown in blue in every panel; peer countries are shown in muted grey tones.

## Repositories, Organizations, and Git Pushes

The same population-normalized comparison is repeated for public repositories, organizations, and git pushes. These series capture different parts of the software-development ecosystem: project stock, institutional presence, and contribution activity.

In [85]:
repositories_chart = make_three_peer_trends(
    repositories,
    raw_col="repositories",
    value_col="repositories_per_100k",
    title="GitHub Repositories per 100k People in Algeria and Peer Countries",
    subtitle="Quarterly public repositories per 100k people, 2020 Q1-2025 Q4",
    y_axis_title="Repositories per 100k people",
)

attaviz.add_caption(repositories_chart, "Source: GitHub Innovation Graph data; World Bank SP.POP.TOTL")

alt.VConcatChart(...)

In [86]:
organizations_chart = make_three_peer_trends(
    organizations,
    raw_col="organizations",
    value_col="organizations_per_100k",
    title="GitHub Organizations per 100k People in Algeria and Peer Countries",
    subtitle="Quarterly GitHub organizations per 100k people, 2020 Q1-2025 Q4",
    y_axis_title="Organizations per 100k people",
)

attaviz.add_caption(organizations_chart, "Source: GitHub Innovation Graph data; World Bank SP.POP.TOTL")

alt.VConcatChart(...)

In [87]:
git_pushes_chart = make_three_peer_trends(
    git_pushes,
    raw_col="git_pushes",
    value_col="git_pushes_per_100k",
    title="Git Pushes per 100k People in Algeria and Peer Countries",
    subtitle="Quarterly git pushes per 100k people, 2020 Q1-2025 Q4",
    y_axis_title="Git pushes per 100k people",
)

attaviz.add_caption(git_pushes_chart, "Source: GitHub Innovation Graph data; World Bank SP.POP.TOTL")

alt.VConcatChart(...)

## Top GitHub Languages

The chart below shows the top 10 languages by number of GitHub pushers for Algeria and each peer country in the latest available quarter. Algeria is highlighted in blue; all peer countries are shown in grey.

In [88]:
clusters = pd.read_csv(GITHUB_DIR / "language_to_cluster_mapping.csv", sep=",")
clusters

,Language,Cluster,Cluster Name
0,AMPL,9,Specialized Tools
1,ANTLR,8,Text Parsing DSLs
2,ASP.NET,22,Enterprise OOP
3,ActionScript,31,Dynamic General Purpose
4,Ada,28,Strongly Typed Systems
...,...,...,...
137,Vue,1,JavaScript-Based Templating
138,XS,19,Low-Level Languages
139,XSLT,18,Low-Level Utilities
140,Yacc,8,Text Parsing DSLs


In [89]:
languages

,iso2_code,country_name,cluster,quarter_start,year,quarter,num_pushers
0,CL,Chile,Batch/High-Level Tools,2025-07-01,2025,3,171
1,CL,Chile,Batch/High-Level Tools,2025-10-01,2025,4,306
2,CL,Chile,Cross-Platform Build Systems,2020-01-01,2020,1,122
3,CL,Chile,Cross-Platform Build Systems,2020-04-01,2020,2,160
4,CL,Chile,Cross-Platform Build Systems,2020-07-01,2020,3,173
...,...,...,...,...,...,...,...
7360,VN,Vietnam,Web Markup/Styling,2024-10-01,2024,4,166896
7361,VN,Vietnam,Web Markup/Styling,2025-01-01,2025,1,150956
7362,VN,Vietnam,Web Markup/Styling,2025-04-01,2025,2,184989
7363,VN,Vietnam,Web Markup/Styling,2025-07-01,2025,3,168791


In [90]:
languages = (
    pd.read_csv(GITHUB_DIR / "languages.csv")
    .loc[lambda d: d["iso2_code"].isin(PEER_CODES)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(),
        country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES),
    )
)
languages['cluster'] = languages['language'].map(clusters.set_index('Language')['Cluster Name'])
languages = languages.groupby(["iso2_code", "country_name", "cluster", 'quarter_start', 'year', 'quarter'])['num_pushers'].sum().reset_index()
latest_language_quarter = languages["quarter_start"].max()
latest_language_rows = languages.loc[languages["quarter_start"] == latest_language_quarter].copy()
latest_language_label = (
    f"{latest_language_rows['year'].iloc[0]} Q{latest_language_rows['quarter'].iloc[0]}"
)

top_languages = (
    latest_language_rows
    .sort_values(["country_name", "num_pushers"], ascending=[True, False])
    .groupby("iso2_code", as_index=False, group_keys=False)
    .head(10)
    .assign(
        rank=lambda d: d.groupby("iso2_code")["num_pushers"].rank(method="first", ascending=False).astype(int),
        num_pushers_label=lambda d: d["num_pushers"].map(lambda v: f"{v:,.0f}"),
    )
)

language_bars = (
    alt.Chart(top_languages)
    .mark_bar(size=11)
    .encode(
        x=alt.X("num_pushers:Q", title="Pushers", axis=alt.Axis(format="~s")),
        y=alt.Y(
            "cluster:N",
            sort=alt.SortField(field="rank", order="ascending"),
            title=None,
            axis=alt.Axis(labelLimit=95),
        ),
        color=alt.condition(
            alt.datum.iso2_code == COUNTRY_CODE,
            alt.value(ALGERIA_COLOR),
            alt.value("#8A969F"),
        ),
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("cluster:N", title="Cluster"),
            alt.Tooltip("cluster:N", title="Type"),
            alt.Tooltip("num_pushers:Q", title="Pushers", format=","),
        ],
    )
)
language_labels = (
    alt.Chart(top_languages)
    .mark_text(align="left", baseline="middle", dx=4, fontSize=9, color="#444")
    .encode(
        x=alt.X("num_pushers:Q"),
        y=alt.Y(
            "cluster:N",
            sort=alt.SortField(field="rank", order="ascending"),
            title=None,
        ),
        text="num_pushers_label:N",
    )
)

languages_chart = (
    (language_bars + language_labels)
    .properties(width=150, height=140)
    .facet(
        facet=alt.Facet(
            "country_name:N",
            title=None,
            sort=[COUNTRY_NAMES[code] for code in PEER_CODES],
            header=alt.Header(labelFontSize=12, labelFontWeight=600),
        ),
        columns=5,
    )
    .resolve_scale(x="independent", y="independent")
    .properties(
        title=alt.Title(
            text="Top GitHub Languages in Algeria and Peer Countries",
            subtitle=f"Top 10 languages by pushers, latest available quarter: {latest_language_label}",
            fontSize=14,
            subtitleFontSize=11,
            offset=10,
        )
    )
)

attaviz.add_caption(languages_chart, "Source: GitHub Innovation Graph languages data")

alt.VConcatChart(...)

## Algeria's Top GitHub Collaborators

GitHub's economic collaborator data records cross-economy collaboration weights. For Algeria, the chart below combines rows where Algeria is either the source or destination and ranks partner economies by total collaboration weight in the latest available quarter.

In [91]:
collaborators = (
    pd.read_csv(GITHUB_DIR / "economy_collaborators.csv")
    .loc[lambda d: (d["source"] == COUNTRY_CODE) | (d["destination"] == COUNTRY_CODE)]
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp(),
        partner_iso2=lambda d: d.apply(
            lambda row: row["destination"] if row["source"] == COUNTRY_CODE else row["source"],
            axis=1,
        ),
    )
)
latest_collab_quarter = collaborators["quarter_start"].max()
latest_collab_label = (
    f"{collaborators.loc[collaborators['quarter_start'] == latest_collab_quarter, 'year'].iloc[0]} "
    f"Q{collaborators.loc[collaborators['quarter_start'] == latest_collab_quarter, 'quarter'].iloc[0]}"
)

top_collaborators = (
    collaborators.loc[lambda d: d["quarter_start"] == latest_collab_quarter]
    .groupby("partner_iso2", as_index=False)["weight"]
    .sum()
    .assign(
        partner_name=lambda d: d["partner_iso2"].map(country_name_from_iso2),
        weight_label=lambda d: d["weight"].map(lambda v: f"{v:,.0f}"),
    )
    .sort_values("weight", ascending=False)
    .head(10)
)

y_sort = top_collaborators["partner_name"].tolist()
collab_bars = (
    alt.Chart(top_collaborators)
    .mark_bar(size=16)
    .encode(
        x=alt.X("weight:Q", title="Collaboration weight", axis=alt.Axis(format="~s")),
        y=alt.Y("partner_name:N", sort=y_sort, title=None),
        color=alt.value(ALGERIA_COLOR),
        tooltip=[
            alt.Tooltip("partner_name:N", title="Partner"),
            alt.Tooltip("partner_iso2:N", title="ISO2"),
            alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
        ],
    )
)
collab_labels = (
    alt.Chart(top_collaborators)
    .mark_text(align="left", baseline="middle", dx=5, fontSize=11, color="#444")
    .encode(
        x=alt.X("weight:Q"),
        y=alt.Y("partner_name:N", sort=y_sort),
        text="weight_label:N",
    )
)

collaborators_chart = (collab_bars + collab_labels).properties(
    width=420,
    height=280,
    title=alt.Title(
        text="Top GitHub Collaborators for Algeria",
        subtitle=f"Latest available quarter: {latest_collab_label}",
        fontSize=14,
        subtitleFontSize=11,
        offset=10,
    ),
)

attaviz.add_caption(collaborators_chart, "Source: GitHub Innovation Graph economy collaborators data")

alt.VConcatChart(...)

## Top GitHub Collaborators for Peer Countries

The same bidirectional collaborator logic can be applied to every peer country. Each panel below shows the top 10 partner economies for one peer country in the latest available quarter.

In [92]:
peer_country_codes = [code for code in PEER_CODES if code != COUNTRY_CODE]

economy_collaborators = (
    pd.read_csv(GITHUB_DIR / "economy_collaborators.csv")
    .assign(
        quarter_start=lambda d: pd.PeriodIndex(
            d["year"].astype(str) + "Q" + d["quarter"].astype(str), freq="Q"
        ).to_timestamp()
    )
)
latest_peer_collab_quarter = economy_collaborators["quarter_start"].max()
latest_peer_collab_rows = economy_collaborators.loc[
    economy_collaborators["quarter_start"] == latest_peer_collab_quarter
].copy()
latest_peer_collab_label = (
    f"{latest_peer_collab_rows['year'].iloc[0]} Q{latest_peer_collab_rows['quarter'].iloc[0]}"
)

peer_partner_rows = []
for focal_iso2 in peer_country_codes:
    focal_rows = latest_peer_collab_rows.loc[
        (latest_peer_collab_rows["source"] == focal_iso2)
        | (latest_peer_collab_rows["destination"] == focal_iso2)
    ].copy()
    focal_rows["focal_iso2"] = focal_iso2
    focal_rows["partner_iso2"] = focal_rows.apply(
        lambda row: row["destination"] if row["source"] == focal_iso2 else row["source"],
        axis=1,
    )
    peer_partner_rows.append(focal_rows)

peer_top_collaborators = (
    pd.concat(peer_partner_rows, ignore_index=True)
    .groupby(["focal_iso2", "partner_iso2"], as_index=False)["weight"]
    .sum()
    .assign(
        focal_name=lambda d: d["focal_iso2"].map(COUNTRY_NAMES),
        partner_name=lambda d: d["partner_iso2"].map(country_name_from_iso2),
        weight_label=lambda d: d["weight"].map(lambda v: f"{v:,.0f}"),
    )
    .sort_values(["focal_name", "weight"], ascending=[True, False])
    .groupby("focal_iso2", as_index=False, group_keys=False)
    .head(10)
    .assign(rank=lambda d: d.groupby("focal_iso2")["weight"].rank(method="first", ascending=False).astype(int))
)

peer_collab_bars = (
    alt.Chart(peer_top_collaborators)
    .mark_bar(size=12)
    .encode(
        x=alt.X("weight:Q", title="Collaboration weight", axis=alt.Axis(format="~s")),
        y=alt.Y(
            "partner_name:N",
            sort=alt.SortField(field="weight", order="descending"),
            title=None,
            axis=alt.Axis(labelLimit=92),
        ),
        color=alt.value("#8A969F"),
        tooltip=[
            alt.Tooltip("focal_name:N", title="Country"),
            alt.Tooltip("partner_name:N", title="Partner"),
            alt.Tooltip("partner_iso2:N", title="Partner ISO2"),
            alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
        ],
    )
)
peer_collab_labels = (
    alt.Chart(peer_top_collaborators)
    .mark_text(align="left", baseline="middle", dx=4, fontSize=9, color="#444")
    .encode(
        x=alt.X("weight:Q"),
        y=alt.Y(
            "partner_name:N",
            sort=alt.SortField(field="weight", order="descending"),
            title=None,
        ),
        text="weight_label:N",
    )
)

peer_collaborators_chart = (
    (peer_collab_bars + peer_collab_labels)
    .properties(width=170, height=150)
    .facet(
        facet=alt.Facet(
            "focal_name:N",
            title=None,
            sort=[COUNTRY_NAMES[code] for code in peer_country_codes],
            header=alt.Header(labelFontSize=12, labelFontWeight=600),
        ),
        columns=3,
    )
    .resolve_scale(x="independent", y="independent")
    .properties(
        title=alt.Title(
            text="Top GitHub Collaborators for Peer Countries",
            subtitle=f"Latest available quarter: {latest_peer_collab_label}",
            fontSize=14,
            subtitleFontSize=11,
            offset=10,
        )
    )
)

attaviz.add_caption(peer_collaborators_chart, "Source: GitHub Innovation Graph economy collaborators data")

alt.VConcatChart(...)